In [2]:
from pngparser import *
import struct
import zlib
import re

In [37]:
parser = PngParser('./intro-forensics-3')
data_chunks = [c.data for c in parser.chunks if c.type == TYPE_IDAT]
data = b''.join(data_chunks)
data = bytearray(data)

In [60]:
def generate_next_depth(current_possibility: list, all_chunks: list):
    data = current_possibility[0]
    zlib_obj = current_possibility[1]
    used_chunk_indices = current_possibility[2]
    new_possibilities = []

    for i, othr in enumerate(all_chunks):
        if i in used_chunk_indices:
            continue

        try:
            new_zlib_obj = zlib_obj.copy()
            addy_data = new_zlib_obj.decompress(othr)
            new_possibility = [data + addy_data, new_zlib_obj, list(used_chunk_indices) + [i]]
            new_possibilities.append(new_possibility)
        except:
            pass

    return new_possibilities

def generate_next_depth_for_all(all_current_possibilities, all_chunks):
    all_new = []

    for current_possibility in all_current_possibilities:
        all_new.extend(generate_next_depth(current_possibility, all_chunks))

    return all_new

In [61]:
def to_pil_img(img):
    from PIL import Image as PilImage

    # raw_data = self.to_bytes()

    color_type = img.header.color_type
    pixel_len = img.header.pixel_len
    pil_pixel_type = 'RGB'

    if color_type == 0:  # Greyscale
        pil_pixel_type = 'L'  # 1
    elif color_type == 2:  # RGB
        pil_pixel_type = 'RGB'  # 3
    elif color_type == 3:  # Palette
        pil_pixel_type = 'RGB'  # 3
        pixel_len = 3  # pixel len in output image must is 3
    elif color_type == 4:  # Greyscale + alpha
        pil_pixel_type = 'LA'  # 2
    elif color_type == 6:  # RGB + Alpha
        pil_pixel_type = 'RGBA'  # 4
    
    pil_img = PilImage.new(pil_pixel_type, (img.header.width + 1, len(img.scanlines)))
    data = []
    
    for sc in img.scanlines:
        filter_pixel = (sc.filter, sc.filter, sc.filter, 1)
        if pixel_len == 1:
            data.append(filter_pixel[0])
            raw_sc = list(map(lambda x: x[0], img._get_pixels(sc.data)))
            if len(raw_sc) < img.header.width:
                raw_sc += [0]*(img.header.width-len(raw_sc))
            data += raw_sc

        else:
            data.append(filter_pixel)
            raw_sc = img._get_pixels(sc.data)
            if len(raw_sc) < img.header.width:
                raw_sc += [(0, 0, 0, 1)]*(img.header.width-len(raw_sc))
            data += raw_sc

    pil_img.putdata(data)
    return pil_img

In [73]:
import tkinter as tk
from PIL import Image, ImageTk

def show_possibility(possibility):
    window = tk.Tk()
    header = parser.get_header()
    img = ImageData(header, possibility[0])
    pil_img = to_pil_img(img)
    img = ImageTk.PhotoImage(pil_img)
    lbl = tk.Label(window, image = img).pack()

    answer = None

    def set_answer(answer_set):
        nonlocal answer
        answer = answer_set
        window.destroy()

    btn_yes = tk.Button(window, text="Yes", command=lambda: set_answer(True)).pack()
    btn_no = tk.Button(window, text="No", command=lambda: set_answer(False)).pack()
    window.mainloop()

    return answer

In [ ]:
current_possibilities = [[b'', zlib.decompressobj(), []]]
for iteration in range(len(data_chunks)):
    if len(current_possibilities) == 0:
        print("Data Exhausted, now possibilities from last round")
        break
    
    current_possibilities = generate_next_depth_for_all(current_possibilities, data_chunks)
    print(f"Iteration #{iteration}, found {len(current_possibilities)} possibilities")

    take_to_next_iteration = []

    for p in current_possibilities:
        if show_possibility(p):
            take_to_next_iteration.append(p)

    print(f"Iteration #{iteration} over, {len(take_to_next_iteration)} survived")
    last_iteration_possibilities = current_possibilities
    current_possibilities = take_to_next_iteration

Iteration #0, found 1 possibilities
Iteration #0 over, 1 survived
Iteration #1, found 11 possibilities
Iteration #1 over, 1 survived
Iteration #2, found 13 possibilities
Iteration #2 over, 1 survived
Iteration #3, found 12 possibilities
Iteration #3 over, 1 survived
Iteration #4, found 11 possibilities
Iteration #4 over, 1 survived
Iteration #5, found 10 possibilities
Iteration #5 over, 1 survived
Iteration #6, found 8 possibilities
Iteration #6 over, 1 survived
Iteration #7, found 7 possibilities
Iteration #7 over, 1 survived
Iteration #8, found 9 possibilities
Iteration #8 over, 1 survived
Iteration #9, found 10 possibilities
Iteration #9 over, 1 survived
Iteration #10, found 8 possibilities
Iteration #10 over, 1 survived
Iteration #11, found 7 possibilities
Iteration #11 over, 1 survived
Iteration #12, found 4 possibilities
Iteration #12 over, 1 survived
Iteration #13, found 7 possibilities
Iteration #13 over, 1 survived
Iteration #14, found 7 possibilities
Iteration #14 over, 1 sur